In [36]:
# input
path2pic = 'reference_images'
path2triplet = 'triplet_dataset'
triplet_dataset = 'triplets_large_final_correctednc_correctedorder.csv'
triplet_index = 'unique_id.txt'
img_extension = '.jpg'

# output
random_index = 'random_index.txt'
triplet_train_index = 'triplet_train_index.csv'
triplet_test_index = 'triplet_test_index.csv'
clip_encoder_index = 'clip_index.npy'
pixel_pca_index = 'pixel_index.npy'

# Load and organize data

In [5]:
import os
root_dir = os.path.dirname(os.getcwd())

pic_dir = os.path.join(root_dir, path2pic)
triplet_data = os.path.join(root_dir, path2triplet, triplet_dataset)
triplet_index = os.path.join(root_dir, path2triplet, triplet_index)
random_index = os.path.join(root_dir, path2triplet, random_index)

## Get picture name

In [6]:
with open(triplet_index) as f:
    picture_names = [line.strip()+img_extension for line in f.readlines()]
picture_names[0:3]

['aardvark.jpg', 'abacus.jpg', 'accordion.jpg']

In [7]:
len(picture_names)

1854

In [24]:
picture_path = [os.path.join(pic_dir, picture_name) for picture_name in picture_names]
picture_path

['/project/wilma/katyzhang/MACS30100/reference_images/aardvark.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/abacus.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/accordion.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/acorn.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/air_conditioner.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/air_mattress.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/air_pump.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/airbag.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/airboat.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/aircraft_carrier.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/airplane.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/album.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/alligator.jpg',
 '/project/wilma/katyzhang/MACS30100/reference_images/almond.jpg',
 '/project/wilma/katyzhang

## Get triplet data

### Extract random sample from whole dataset
Randomly select 300,000 data, cause the dataset is too large: 469,9160

In [8]:
sample_n = 300000

In [9]:
import pandas as pd

# read data
triplet = pd.read_csv(triplet_data, sep="\t")
triplet.head(5)

,image1,image2,image3,choice,RT,noise_ceiling,subject_id,HIT_nr,trial_nr,age,gender,date,time,dataset
0,1245,1050,494,3,14457,0,BMWLG5ZY79VLA,1,1,NaN,other,2018-02-05,09:31:16,1
1,888,1788,1250,3,5043,0,BMWLG5ZY79VLA,1,2,NaN,other,2018-02-05,09:31:16,1
2,1256,734,946,2,6605,0,BMWLG5ZY79VLA,1,3,NaN,other,2018-02-05,09:31:16,1
3,1069,1037,953,1,6177,0,BMWLG5ZY79VLA,1,4,NaN,other,2018-02-05,09:31:16,1
4,1693,803,963,3,2327,0,BMWLG5ZY79VLA,1,5,NaN,other,2018-02-05,09:31:16,1


In [10]:
import numpy as np

# remove the duplicate
cols = ["image1", "image2", "image3"]

# create an order-invariant key by sorting the three IDs row-wise
key = pd.DataFrame(np.sort(triplet[cols].to_numpy(), axis=1), columns=[f"k{i}" for i in range(3)])

triplet_unique_any_order = triplet.loc[~key.duplicated()].reset_index(drop=True)

In [11]:
len(triplet_unique_any_order)

4567526

In [ ]:
# # Randomly select 100,000 data, cause the dataset is too large: 469,9160
# all_images_included_tag = 0
# no_duplicate_tag = 0
# while not (all_images_included_tag and no_duplicate_tag):
#     random_index_label = triplet_unique_any_order.sample(n=sample_n).index
#     # to make sure that all the images are included
#     image_set = set() 
#     # detect duplicate
#     duplicate_detect_set = set()
#     for index in random_index_label:
#         img1 = triplet_unique_any_order["image1"].iloc[index]
#         img2 = triplet_unique_any_order["image2"].iloc[index]
#         img3 = triplet_unique_any_order["image3"].iloc[index]
#         image_set.add(img1)
#         image_set.add(img2)
#         image_set.add(img3)
#         duplicate_detect_set.add((img1, img2, img3))

#     print(random_index_label.unique())
#     print(len(image_set))
#     all_images_included_tag = (len(image_set)==len(picture_names))
#     print(len(duplicate_detect_set))
#     no_duplicate_tag = (len(duplicate_detect_set)==len(random_index_label))

Index([3996427, 1162552, 1843007,  496232, 2283699, 3373103, 2738035, 4142925,
       3268663,  353572,
       ...
        880754, 3805028,  816803, 1918890, 1013648, 1530268,   30022, 2122434,
       4194649, 1519457],
      dtype='int64', length=300000)
1854
300000


In [ ]:
# print("All images (1854) are included in the sample:", all_images_included_tag)
# print("No duplicate images combinations in the sample:", no_duplicate_tag)

All images (1854) are included in the sample: True
No duplicate images combinations in the sample: True


In [ ]:
# with open(random_index, mode = "w") as f:
#     for index in random_index_label:
#         f.write(str(index) + '\n')

In [12]:
with open(random_index) as f:
    index_selected = [line.strip() for line in f.readlines()]

In [13]:
len(index_selected)

300000

In [14]:
triplet_sample = triplet_unique_any_order[["image1", "image2", "image3", "choice"]].iloc[index_selected]
triplet_sample.head(5)

,image1,image2,image3,choice
3996427,1610,929,1348,1
1162552,875,1398,1663,2
1843007,585,877,1705,2
496232,1641,688,438,1
2283699,63,834,1473,2


### Split the data into trining and testing test
Cause the training data would be permutated, yet the test data would not be

In [15]:
from sklearn.model_selection import train_test_split

triplet_sample_X = triplet_sample[["image1", "image2", "image3"]]
triplet_sample_y = triplet_sample["choice"]
X_train, X_test, y_train, y_test = train_test_split(triplet_sample_X, triplet_sample_y, test_size=0.3, random_state=42)
df_train = X_train.join(y_train)
df_test = X_test.join(y_test)

In [16]:
df_test

,image1,image2,image3,choice
1854915,656,1141,42,2
2732853,1611,1788,374,1
91147,362,87,1750,1
2347270,367,783,1572,2
10491,803,1665,630,2
...,...,...,...,...
4453839,662,334,255,1
1897231,356,1717,869,1
4547346,1301,247,643,1
2194314,770,1140,640,1


In [17]:
# store the test data
df_test.to_csv(triplet_test_index)

### Permutation of a, b, and c
In this project, each sample is defined as a triplet of a, b, and c, representing the items in odd-one-out judgment. They function as positional indices used to distinguish the three slots within a triplet, instead of a new set of stimuli. 
Changing the ordering of samples aims to avoid the model’s learning of positional biases, or treating the position itself as informative. To prevent this, we permute the order of a, b, and c during the data processing and exploratory data analysis. The permutations of the same triplet represent an equivalent set (e.g., (a, b, c), (b, a, c), (c, b, a)) and should yield the same interpretations.

In [20]:
df_train

,image1,image2,image3,choice
3328352,675,416,1522,1
2280671,947,1314,337,2
1509646,1323,978,684,2
4310536,1235,954,103,2
4193715,1386,655,760,3
...,...,...,...,...
263709,1424,1585,1260,2
787121,1736,411,1305,2
2480773,1694,964,1777,1
4273037,636,1259,535,1


In [19]:
from itertools import permutations

def build_perm_dict():
    perm_dict = {} # Initailize the dictionary
    # 6 list, each value represent the original element's position
    perms = list(permutations([1, 2, 3]))
    # print(perms)

    # Loop: each choice
    for old_choice in [1, 2, 3]:
        out = []
        # Loop: through the permutation
        for p in perms:
            # value in p represents the original element's position
            new_choice = p.index(old_choice) + 1 # to avoid the problem of 0 when training the model
            out.append((p[0], p[1], p[2], new_choice)) # append the data
        perm_dict[str(old_choice)] = out
    return perm_dict

perm_dict = build_perm_dict()
perm_dict["2"]

[(1, 2, 3, 2),
 (1, 3, 2, 3),
 (2, 1, 3, 1),
 (2, 3, 1, 1),
 (3, 1, 2, 3),
 (3, 2, 1, 2)]

In [ ]:
# import pandas as pd

# def apply_perm_expand(df, perm_dict):
#     rows = []
#     # Loop: for each row in dataframe
#     for _, r in df.iterrows():
#         # only care about 4 columns from the original dataframe
#         r = r[["image1", "image2", "image3", "choice"]]
#         c = str(int(r["choice"]))  # get choice: "1"/"2"/"3"
#         # Loop: through perm_dict
#         for (A_, B_, C_, new_c) in perm_dict[c]:
#             # reorganize
#             old_imgs = [r["image1"], r["image2"], r["image3"]]
#             new_imgs = [old_imgs[A_-1], old_imgs[B_-1], old_imgs[C_-1]]

#             # store in the right order
#             new_r = r.copy()
#             new_r["image1"], new_r["image2"], new_r["image3"] = new_imgs
#             new_r["choice"] = new_c

#             # record the permutation
#             new_r["perm"] = (A_, B_, C_)
#             rows.append(new_r)

#     return pd.DataFrame(rows).reset_index(drop=True)

# triplet_perm = apply_perm_expand(df_train, perm_dict)

In [ ]:
# triplet_perm.to_csv(triplet_train_index, index=False)

In [38]:
triplet_perm = pd.read_csv(triplet_train_index)
print(len(triplet_perm))
triplet_perm.head(10)

1260000


,image1,image2,image3,choice,perm
0,675,416,1522,1,"(1, 2, 3)"
1,675,1522,416,1,"(1, 3, 2)"
2,416,675,1522,2,"(2, 1, 3)"
3,416,1522,675,3,"(2, 3, 1)"
4,1522,675,416,2,"(3, 1, 2)"
5,1522,416,675,3,"(3, 2, 1)"
6,947,1314,337,2,"(1, 2, 3)"
7,947,337,1314,3,"(1, 3, 2)"
8,1314,947,337,1,"(2, 1, 3)"
9,1314,337,947,1,"(2, 3, 1)"


## Convert pictures using CLIP encoder
Using [CLIP](https://openai.com/index/clip/) to convert one image into a vector (1, 512) to represent this image's positions in CLIP "semantic space".
- In other word, embedding the pictures using the semantic information

In [ ]:
import clip
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load(
    "ViT-B/32",
    device=device,
    download_root="/project/wilma/katyzhang/CLIP"
)
model.eval()

In [33]:
import PIL.Image as Image
import json

def encode_image_openclip(image_paths, device=device, dtype=np.float32):
    N = len(image_paths)
    X = np.zeros((N, 512), dtype=dtype)

    for i, p in enumerate(image_paths):
        image = Image.open(p).convert("RGB")
        
        image_tensor = preprocess(image).unsqueeze(0).to(device)
        with torch.no_grad():
            image_features = model.encode_image(image_tensor)
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)

        if (i + 1) % 200 == 0:
            print(f"Loaded {i+1}/{N}")

        X[i] = image_features
    
    return X

clip_index = encode_image_openclip(picture_path)
print("")
print("clip_index:", clip_index.shape)

Loaded 200/1854
Loaded 400/1854
Loaded 600/1854
Loaded 800/1854
Loaded 1000/1854
Loaded 1200/1854
Loaded 1400/1854
Loaded 1600/1854
Loaded 1800/1854

clip_index: (1854, 512)


In [37]:
np.save(clip_encoder_index, clip_index)

## Convert pictures into pixels
To match the output dimension of CLIP (512,), we use PCA to capture the picture's pixel level charateristics. To avoid memory issue, we would resize the pictures into (64, 64, 3)
- In other word, embedding the pictures using the perceptual information

In [ ]:
import json
import numpy as np
from PIL import Image
from sklearn.decomposition import PCA

def load_pixels_matrix(image_paths, size=(800, 800), dtype=np.float32):
    """
    resize the pictures into (64, 64,3)
    """
    N = len(image_paths)
    D = size[0] * size[1] * 3
    X = np.zeros((N, D), dtype=dtype)

    for i, p in enumerate(image_paths):
        img = Image.open(p).convert("RGB").resize(size, Image.BICUBIC)
        arr = np.asarray(img, dtype=np.uint8)               # (64,64,3)
        X[i] = (arr.reshape(-1).astype(dtype) / 255.0)      # (12288,)

        if (i + 1) % 200 == 0:
            print(f"Loaded {i+1}/{N}")

    return X

# ---- usage ----
# image_paths = [...]  # list of 1854 file paths

X = load_pixels_matrix(picture_path, size=(800, 800))

pca = PCA(n_components=512, random_state=0)
pixel_index = pca.fit_transform(X)   # (N,512)

print("")
print("X:", X.shape, "pixel_index:", pixel_index.shape)

Loaded 200/1854
Loaded 400/1854
Loaded 600/1854
Loaded 800/1854
Loaded 1000/1854
Loaded 1200/1854
Loaded 1400/1854
Loaded 1600/1854
Loaded 1800/1854


: 

In [ ]:
np.save(pixel_pca_index, pixel_index)